# Knowlytix GMS — keyless demo quickstart

This notebook runs **with no license key and no LLM API key**. `knowlytix-core` drops to keyless *demo mode* (built-in fixtures only), so you can explore the geometric knowledge graph, score benchmark questions, and call the LLM adapter — all offline.

Get a free 180-day developer key at [knowlytix.ai/signup](https://knowlytix.ai/signup) to use your own data.

In [ ]:
# Demo mode: no key required. Set BEFORE importing knowlytix.core.
import os
os.environ.setdefault("KNOWLYTIX_DEMO_MODE", "1")
os.environ.setdefault("KNOWLYTIX_QUIET", "1")

import knowlytix.core as core
assert core.IS_DEMO_MODE, "expected keyless demo mode"
print("demo mode:", core.IS_DEMO_MODE)

## 1. The bundled demo corpus
A small credit-risk dataset ships inside `knowlytix-benchmark`.

In [ ]:
from knowlytix.benchmark.fixtures.demo import load

corpus = load()
print("artifacts:", sorted(corpus))
print("document:", corpus["document"]["title"])
print("questions:", len(corpus["questions"]))

## 2. Query the geometric knowledge graph
Reconstruct the bundled `GeometricKnowledgeGraph` and rank candidate tails for a `(head, relation, ?)` query — a real geometric read, no training required.

In [ ]:
import pickle
import torch
from knowlytix.core.graph.gkg import GeometricKnowledgeGraph

with open(corpus["knowledge_graph"], "rb") as fh:
    blob = pickle.load(fh)

kg = GeometricKnowledgeGraph(blob["num_entities"], blob["num_relations"])
kg.load_state_dict(blob["state_dict"])
kg.eval()
entities, relations = blob["entities"], blob["relations"]

h, r, _ = blob["facts"][0]
with torch.no_grad():
    scores = kg.score_all_tails(torch.tensor([h]), torch.tensor([r]))
top = torch.topk(scores[0], 3, largest=False).indices.tolist()
print(f"{entities[h]} --{relations[r]}--> ?")
print("  top tails:", [entities[i] for i in top])

## 3. Score the pre-scored benchmark questions
`score_answer` is domain-agnostic and runs offline.

In [ ]:
from knowlytix.benchmark import score_answer

for q in corpus["questions"]:
    result = score_answer(q["demo_answer"], q["ground_truth"])
    flag = "ok" if result.correct else "x"
    print(f"[{flag}] {q['id']}: {result.detail}")

## 4. Call the LLM adapter — no API key
In demo mode `get_llm()` returns a `DemoLLMClient` that serves canned responses for bundled prompts (and a labeled placeholder otherwise). The real LLM code path runs; no network call happens.

In [ ]:
from knowlytix.core import get_llm

prompt = "In one sentence, what does a borrower's credit score tell an underwriter about default risk?"
reply = get_llm().complete([{"role": "user", "content": prompt}])
print(reply)

## 5. The demo boundary
Demo mode is read-only: ingesting your own data or training raises `DemoModeError` pointing to signup. This is the gate, shown as a teaching moment — a developer key unlocks these paths.

In [ ]:
from knowlytix.core import DemoModeError
from knowlytix.core.memory.enm import ExactNumericalMemory, ENMKey

try:
    ExactNumericalMemory().write(ENMKey("borrower", "A"), 720.0)
    print("wrote (unexpected — are you licensed?)")
except DemoModeError as exc:
    print("blocked in demo mode:", exc)

## Next steps
- Get a free **developer** key (180 days, your own data): [knowlytix.ai/signup](https://knowlytix.ai/signup)
- Then unset `KNOWLYTIX_DEMO_MODE` and set `KNOWLYTIX_LICENSE_KEY` — the same notebooks run against real data.